# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cokezero20/FlyRank_AI_ML_Internship_NATIVIDAD/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** One row = (content_hash_id, report_date)
- Each row represents one content item on one day

**Time windows for feature engineering:**
- Feature window: [report_date - 90 days, report_date - 1 day]
- Label window: [report_date + 1 day, report_date + 30 days]
- We compute these from raw daily data using Jan-Apr 2026 (90-day history + 30-day future labels)

In [4]:
from datasets import load_dataset
from google.colab import userdata
import pandas as pd
from datetime import date

HF_TOKEN = userdata.get('HF_Token')

dataset = load_dataset(
    'FlyRank/internship-warehouse',
    name='fact_content_daily_performance',
    token=HF_TOKEN,
    streaming=True
)

train_split = dataset['train']

print("Loading January-April 2026 data...")
data_rows = []
batch_size = 100000
batch_count = 0

for batch in train_split.iter(batch_size=batch_size):
    batch_count += 1
    batch_df = pd.DataFrame(batch)

    # Ensure report_date is date object
    if isinstance(batch_df['report_date'].iloc[0], str):
        batch_df['report_date'] = pd.to_datetime(batch_df['report_date']).dt.date

    # Filter for Jan-Apr 2026 and ga4_data_available = True (as per skill)
    data_batch = batch_df[
        (batch_df['report_date'] >= date(2026, 1, 1)) &
        (batch_df['report_date'] <= date(2026, 4, 30)) &
        (batch_df['ga4_data_available'] == True)
    ]

    if len(data_batch) > 0:
        data_rows.append(data_batch)
        print(f"  Batch {batch_count}: {len(data_batch):,} rows")

df_4months = pd.concat(data_rows, ignore_index=True)
print(f"\n✓ Total: {len(df_4months):,} rows")

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading January-April 2026 data...
  Batch 200: 1,046 rows
  Batch 201: 702 rows
  Batch 202: 1,998 rows
  Batch 203: 1,008 rows
  Batch 204: 2,250 rows
  Batch 205: 1,597 rows
  Batch 206: 3,844 rows
  Batch 207: 351 rows
  Batch 208: 2,704 rows
  Batch 209: 1,323 rows
  Batch 210: 2,163 rows
  Batch 211: 1,730 rows
  Batch 212: 1,371 rows
  Batch 213: 1,721 rows
  Batch 214: 1,673 rows
  Batch 215: 594 rows
  Batch 216: 357 rows
  Batch 217: 1,035 rows
  Batch 218: 775 rows
  Batch 219: 1 rows
  Batch 220: 2,154 rows
  Batch 221: 59 rows
  Batch 222: 3,539 rows
  Batch 223: 2,723 rows
  Batch 224: 378 rows
  Batch 225: 896 rows
  Batch 226: 3,179 rows
  Batch 227: 1,314 rows
  Batch 228: 555 rows
  Batch 229: 2,693 rows
  Batch 230: 683 rows
  Batch 231: 2,031 rows
  Batch 232: 1,499 rows
  Batch 233: 1,270 rows
  Batch 234: 10 rows
  Batch 235: 500 rows
  Batch 236: 2,793 rows
  Batch 237: 931 rows
  Batch 238: 2,195 rows
  Batch 239: 1,294 rows
  Batch 240: 1,721 rows
  Batch 241: 

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features** (raw daily metrics, will compute rolling windows from these):
- gsc_impressions, gsc_clicks, gsc_avg_position
- ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions, ga4_total_engagement_sec
- sessions_organic, sessions_direct, sessions_referral, sessions_social, sessions_paid, sessions_ai
- ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other
- scroll_events

**New features to compute:**
- impressions_prev_30d, impressions_prev_60d, impressions_prev_90d (rolling sums)
- clicks_prev_30d, clicks_prev_60d, clicks_prev_90d (rolling sums)
- (and other rolling windows from raw metrics)

**Label** (to be computed):
- is_declining_label (1 if gsc_impressions drop >20% from March to April, 0 otherwise)

**Context** (grouping/monitoring, never features):
- content_hash_id (unit of analysis)
- client_hash_id (fairness monitoring)
- report_date (time windowing)
- client_has_gsc, client_has_ga4 (data availability flags)

**Excluded** (why):
- gsc_data_available, ga4_data_available (filtering criteria, not features)
- gsc_sum_position (sum of positions not meaningful; use average instead)

In [ ]:
# Verify columns exist
features = ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews',
            'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec',
            'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social',
            'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini',
            'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']

context = ['content_hash_id', 'client_hash_id', 'report_date', 'client_has_gsc', 'client_has_ga4']

excluded = ['gsc_data_available', 'ga4_data_available', 'gsc_sum_position']

print("Features available:", sum(1 for f in features if f in df_4months.columns), f"/ {len(features)}")
print("Context available:", sum(1 for c in context if c in df_4months.columns), f"/ {len(context)}")
print("Excluded available:", sum(1 for e in excluded if e in df_4months.columns), f"/ {len(excluded)}")

# Check for unexpected columns
all_categorized = set(features + context + excluded)
unexpected = set(df_4months.columns) - all_categorized
if unexpected:
    print(f"\nUnexpected columns: {unexpected}")

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

- **Grain:** (content_hash_id, report_date) uniqueness
- **Counts:** Rows per month
- **Missing values:** NULL counts in key columns
- **Windows:** Date coverage for computing 90-day features and 30-day labels

In [ ]:
# 1. Grain verification
print("1. GRAIN VERIFICATION")
grain = df_4months.groupby(['content_hash_id', 'report_date']).size()
duplicates = (grain > 1).sum()
print(f"   Unique (content_hash_id, report_date) pairs: {len(grain):,}")
print(f"   Duplicates: {duplicates}")
print(f"   ✓ Grain OK\n" if duplicates == 0 else f"   ✗ {duplicates} duplicates\n")

# 2. Counts by month
print("2. ROW COUNTS BY MONTH")
df_4months['month'] = df_4months['report_date'].dt.to_period('M')
monthly_counts = df_4months.groupby('month').size()
print(monthly_counts)
print()

# 3. Missing values in key columns
print("3. MISSING VALUES")
key_cols = ['gsc_impressions', 'gsc_clicks', 'ga4_pageviews', 'ga4_sessions']
for col in key_cols:
    missing_pct = (df_4months[col].isna().sum() / len(df_4months) * 100)
    print(f"   {col}: {df_4months[col].isna().sum():,} NULLs ({missing_pct:.2f}%)")
print()

# 4. Date windows
print("4. DATE WINDOWS")
print(f"   Min date: {df_4months['report_date'].min().date()}")
print(f"   Max date: {df_4months['report_date'].max().date()}")
print(f"   Days span: {(df_4months['report_date'].max() - df_4months['report_date'].min()).days} days")
print(f"   ✓ Covers Jan 1 - Apr 30 (enough for 90-day features + 30-day labels)")

In [ ]:
print(f"df_4months shape: {df_4months.shape}")
print(f"df_4months columns: {df_4months.columns.tolist()}")
print(f"First 5 rows:")
print(df_4months.head())

In [ ]:
print(f"Before ga4 filter: {len(data_rows)} batches")
print(f"Total rows before filter:")
df_temp = pd.concat(data_rows, ignore_index=True)
print(f"  {len(df_temp):,} rows")

print(f"\nga4_data_available value counts:")
print(df_temp['ga4_data_available'].value_counts(dropna=False))

print(f"\nRows where ga4_data_available == True: {(df_temp['ga4_data_available'] == True).sum():,}")

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.